In [0]:
%run "/Workspace/Users/dnp50022@gmail.com/Retail-Sales-Data-Pipeline/databricks/notebooks/silver/service principle"

In [0]:

# COMMAND ----------

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import (col, trim, to_date, row_number, current_timestamp, lit,concat_ws,coalesce)

# COMMAND ----------


storage_account = "salesstorageproject"
container_name = "sales"
table_name = "products"

bronze_path = f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/bronze data/{table_name}/*"


In [0]:

# Read Transaction CSV from Bronze
df = spark.read.format("csv").option("header", "true").load(bronze_path)  
display(df)

In [0]:
df.columns

In [0]:
#  CAST DATATYPES
df = (
    df.withColumn("ProductID", col("ProductID").cast("string"))
      .withColumn("ProductName", col("ProductName").cast("string"))
      .withColumn("Category", col("Category").cast("string"))
      .withColumn("Price", col("Price").cast("double"))
      .withColumn("ModifiedDate", to_timestamp(col("ModifiedDate")))
)

In [0]:

#  PK FILTER (ProductID must be valid)
df = df.filter(
    col("ProductID").isNotNull() &
    (trim(col("ProductID")) != "") &
    (col("ProductID") != "0")
)

In [0]:
# BUSINESS RULES
df = df.filter(col("Price") >= 0)

In [0]:
# STANDARDIZE TEXT 
df = df.withColumn("ProductName", initcap(trim(col("ProductName")))) \
       .withColumn("Category", initcap(trim(col("Category"))))


In [0]:
# DEDUPLICATION (Keep Latest Record)
w = Window.partitionBy("ProductID").orderBy(
    col("ModifiedDate").desc()
)

df = (
    df.withColumn("row_num", row_number().over(w))
      .filter(col("row_num") == 1)
      .drop("row_num")
)

In [0]:

# ADD DERIVED COLUMN (optional but useful 🚀)
df = df.withColumn(
    "price_category",
    when(col("Price") > 500, "HIGH")
    .when(col("Price") > 200, "MEDIUM")
    .otherwise("LOW")
)

In [0]:
# ADD INGESTION TIMESTAMP
df = df.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
# WRITE TO UNITY CATALOG
silver_table = "sales.silver.products"

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table)

print("Silver table created:", silver_table)

In [0]:
%sql
select * from sales.silver.products